# SageMaker Fraud Detection Pipeline - Execution Notebook

This notebook allows you to:
1. Create/update the SageMaker pipeline
2. Start pipeline executions
3. Monitor execution progress
4. View results and metrics
5. Manage pipeline versions

**Environment:** Run this in a SageMaker AI Notebook instance

## 1. Setup and Configuration

In [ ]:
# Install project in editable mode so changes to source code are immediately available
! uv pip install -e ../

### Verify training data is ready

Confirms the Kaggle dataset is downloaded, uploaded to S3, and seeded into the Athena `training_data` table — and that the labels are correlated with the features (cheap sanity check on a known-predictive PCA component).

**Idempotent:** if the table is already healthy this cell is a few-second no-op. If it's missing or corrupted, the cell re-downloads from Kaggle, re-uploads to S3, and re-seeds Athena before continuing. Pass `force=True` to always re-seed.


In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path so `src.*` imports work
# even when the kernel's CWD is notebooks/ and the editable install
# hasn't been refreshed for this kernel session.
_project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from src.setup.download_kaggle_dataset import ensure_training_data_ready

ensure_training_data_ready()  # add force=True to force a full re-seed


In [ ]:
import os
import sys
import boto3
from sagemaker.core.helper.session_helper import Session
from pathlib import Path
from dotenv import load_dotenv
from src.utils.aws_utils import get_execution_role

# Determine project root and load .env
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir
env_path = project_root / '.env'

# Load .env file with override=True to ensure values are loaded
if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f"✓ Loaded environment from: {env_path}\n")
else:
    print(f"⚠ .env file not found at: {env_path}\n")

# Initialize AWS clients
region = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
sagemaker_client = boto3.client('sagemaker', region_name=region)
s3_client = boto3.client('s3', region_name=region)

# Get SageMaker session and role using project's wrapper
# This handles fallback to .env and provides better error messages
role = get_execution_role()

sagemaker_session = Session()
default_bucket = sagemaker_session.default_bucket()

# Display configuration
print(f"AWS Configuration:")
print(f"  Region: {region}")
print(f"  Role: {role}")
print(f"  Default S3 bucket: {default_bucket}")

# MLflow Tracking Configuration
mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
mlflow_experiment = os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection')

print(f"\nMLflow Configuration:")
if mlflow_uri:
    print(f"  Tracking URI: {mlflow_uri}")
    print(f"  Experiment: {mlflow_experiment}")
    print(f"\n✓ MLflow tracking is ENABLED")
    print(f"  Training metrics will be logged to MLflow")
    print(f"  View experiments at: SageMaker Console → MLflow")
else:
    print(f"  Tracking URI: Not set in .env")
    print(f"\n⚠ MLflow tracking is NOT configured")
    print(f"  Add MLFLOW_TRACKING_URI to .env file to enable tracking")
    print(f"  Example: MLFLOW_TRACKING_URI=arn:aws:sagemaker:us-east-1:123456789012:mlflow-app/app-XXXXX")

In [ ]:
# Pipeline configuration
PIPELINE_NAME = "fraud-detection-pipeline"
PIPELINE_DESCRIPTION = "Fraud detection ML pipeline with training, evaluation, and deployment"

# Pipeline parameter OVERRIDES for this execution.
# Hyperparameter defaults (MaxDepth, LearningRate, NumBoostRound,
# MinChildWeight, EarlyStoppingRounds) and quality-gate defaults
# (MinRocAuc, MinPrAuc) live in src/config/config.yaml under
# training.xgboost_params and are wired through pipeline.py — leave them
# out here so the YAML stays the single source of truth.
# Override one in PIPELINE_PARAMS only if you want to deviate from YAML
# for a specific run.
PIPELINE_PARAMS = {
    # Data
    'AthenaTable': 'training_data',

    # Deployment
    'EndpointName': 'fraud-detector-endpoint',
    'EndpointMemorySize': '2048',
    'EndpointMaxConcurrency': '10',
    'EnableAthenaLogging': 'true',
    'TestNumSamples': '50',
    'ModelApprovalStatus': 'Approved',
    'ModelPackageGroup': 'fraud-detection',
}

print("Pipeline Configuration:")
print(f"  Name: {PIPELINE_NAME}")
print(f"  Description: {PIPELINE_DESCRIPTION}")
print("\nXGBoost hyperparameters: from src/config/config.yaml (training.xgboost_params)")
print("Quality gates: from pipeline.py defaults (MinRocAuc=0.70, MinPrAuc=0.30)")
print("\nExecution-level overrides:")
for key, value in PIPELINE_PARAMS.items():
    print(f"  {key}: {value}")


## 3. Create/Update Pipeline

This will create the pipeline definition in SageMaker using the fixed `train.py` code.

In [ ]:
# 1. Delete the old pipeline. If you see updates are not getting refreshed, this will refresh the code
try:
    sagemaker_client.delete_pipeline(PipelineName=PIPELINE_NAME)
    print("✓ Pipeline deleted")
except sagemaker_client.exceptions.ResourceNotFound:
    print("ℹ Pipeline does not exist (already deleted or never created)")
except Exception as e:
    print(f"⚠ Error deleting pipeline: {e}")

In [ ]:
from src.train_pipeline.pipeline import create_fraud_detection_pipeline

print("Creating/updating pipeline...\n")

# Display MLflow integration info
mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
if mlflow_uri:
    print("MLflow Integration:")
    print(f"  ✓ Training step will log to: {mlflow_uri}")
    print(f"  ✓ Experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
    print(f"  ✓ Model registry: {os.getenv('MLFLOW_MODEL_NAME', 'xgboost-fraud-detector')}")
    print()

try:
    # Create pipeline builder
    pipeline_builder = create_fraud_detection_pipeline(
        pipeline_name=PIPELINE_NAME,
        region=region,
        role=role
    )
    
    # Upsert pipeline (create if doesn't exist, update if it does)
    result = pipeline_builder.upsert_pipeline(
        description=PIPELINE_DESCRIPTION,
        include_deployment=True,  # Set to False to skip deployment steps
        tags=[
            {'Key': 'Project', 'Value': 'FraudDetection'},
            {'Key': 'Environment', 'Value': 'Production'},
            {'Key': 'ManagedBy', 'Value': 'MLflow'},
        ]
    )
    
    print("✓ Pipeline created/updated successfully!")
    print(f"\nPipeline ARN: {result['pipeline_arn']}")
    
except Exception as e:
    print(f"✗ Failed to create pipeline: {e}")
    import traceback
    traceback.print_exc()

## 4. Start Pipeline Execution

This will start a new execution of the pipeline with the fixed training code.

In [ ]:
# Generate execution name with timestamp
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
execution_name = f"{PIPELINE_NAME}-{timestamp}"

print(f"Starting pipeline execution: {execution_name}\n")

# Format parameters for SageMaker
pipeline_parameters = [
    {'Name': key, 'Value': str(value)}
    for key, value in PIPELINE_PARAMS.items()
]

try:
    response = sagemaker_client.start_pipeline_execution(
        PipelineName=PIPELINE_NAME,
        PipelineExecutionDisplayName=execution_name,
        PipelineParameters=pipeline_parameters
    )
    
    execution_arn = response['PipelineExecutionArn']
    
    print("✓ Pipeline execution started successfully!")
    print(f"\nExecution Details:")
    print(f"  ARN: {execution_arn}")
    print(f"  Name: {execution_name}")
    
    # MLflow logging information
    mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
    if mlflow_uri:
        print(f"\nMLflow Tracking:")
        print(f"  📊 Training metrics are being logged to MLflow")
        print(f"  📈 Tracking Server: {mlflow_uri}")
        print(f"  🔬 Experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
        print(f"  📦 Model Registry: {os.getenv('MLFLOW_MODEL_NAME', 'xgboost-fraud-detector')}")
        print(f"\n  View in MLflow UI:")
        print(f"    1. Open SageMaker Console → MLFlow")
        print(f"    2. Select your MLflow app")
        print(f"    3. Find experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
    else:
        print(f"\n⚠ MLflow tracking not configured")
    
    print(f"\nYou can monitor this execution in the next cell")
    
    # Store for monitoring
    CURRENT_EXECUTION_ARN = execution_arn
    
except Exception as e:
    print(f"✗ Failed to start execution: {e}")
    import traceback
    traceback.print_exc()

## 5. Monitor Pipeline Execution

Watch the pipeline execution in real-time.

In [ ]:
# Generate execution name with timestamp
# from datetime import datetime
# timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
# execution_name = f"{PIPELINE_NAME}-{timestamp}"

# print(f"Starting pipeline execution: {execution_name}\n")

# # Format parameters for SageMaker
# pipeline_parameters = [
#     {'Name': key, 'Value': str(value)}
#     for key, value in PIPELINE_PARAMS.items()
# ]

# try:
#     response = sagemaker_client.start_pipeline_execution(
#         PipelineName=PIPELINE_NAME,
#         PipelineExecutionDisplayName=execution_name,
#         PipelineParameters=pipeline_parameters
#     )
    
#     execution_arn = response['PipelineExecutionArn']
    
#     print("✓ Pipeline execution started successfully!")
#     print(f"\nExecution Details:")
#     print(f"  ARN: {execution_arn}")
#     print(f"  Name: {execution_name}")
    
#     # MLflow logging information
#     mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
#     if mlflow_uri:
#         print(f"\nMLflow Tracking:")
#         print(f"  📊 Training metrics are being logged to MLflow")
#         print(f"  📈 Tracking Server: {mlflow_uri}")
#         print(f"  🔬 Experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
#         print(f"  📦 Model Registry: {os.getenv('MLFLOW_MODEL_NAME', 'xgboost-fraud-detector')}")
#         print(f"\n  View in MLflow UI:")
#         print(f"    1. Open SageMaker Console → MLFlow")
#         print(f"    2. Select your MLflow app")
#         print(f"    3. Find experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
#     else:
#         print(f"\n⚠ MLflow tracking not configured")
    
#     print(f"\nYou can monitor this execution in the next cell")
    
#     # Store for monitoring
#     CURRENT_EXECUTION_ARN = execution_arn
    
# except Exception as e:
#     print(f"✗ Failed to start execution: {e}")
#     import traceback
#     traceback.print_exc()

In [ ]:
  # Check actual model performance from evaluation report                                                                                                                                                                                                         
def get_actual_metrics(execution_arn):                                                                                                                                                                                                                          
      """Get actual metrics from the evaluation report."""                                                                                                                                                                                                        
      import boto3
      import json

      sagemaker_client = boto3.client('sagemaker', region_name=AWS_DEFAULT_REGION)
      s3_client = boto3.client('s3', region_name=AWS_DEFAULT_REGION)

      # Get execution steps
      response = sagemaker_client.list_pipeline_execution_steps(
          PipelineExecutionArn=execution_arn
      )

      steps = response.get('PipelineExecutionSteps', [])

      # Find evaluation step
      for step in steps:
          if 'EvaluateModel' in step['StepName']:
              if 'Metadata' in step and 'ProcessingJob' in step['Metadata']:
                  job_name = step['Metadata']['ProcessingJob']['Arn'].split('/')[-1]

                  # Get processing job details
                  response = sagemaker_client.describe_processing_job(
                      ProcessingJobName=job_name
                  )

                  # Find evaluation output
                  for output in response['ProcessingOutputConfig']['Outputs']:
                      if output['OutputName'] == 'evaluation':
                          s3_uri = output['S3Output']['S3Uri']

                          # Parse S3 URI
                          parts = s3_uri.replace('s3://', '').split('/', 1)
                          bucket = parts[0]
                          key = parts[1] + '/evaluation.json'

                          print(f"Fetching evaluation report from:")
                          print(f"  s3://{bucket}/{key}\n")

                          try:
                              # Download evaluation report
                              obj = s3_client.get_object(Bucket=bucket, Key=key)
                              property_data = json.loads(obj['Body'].read())

                              print("="*80)
                              print("ACTUAL MODEL PERFORMANCE:")
                              print("="*80)

                              metrics = property_data.get('binary_classification_metrics', {})
                              for metric_name, metric_value in metrics.items():
                                  value = metric_value.get('value', 'N/A')
                                  print(f"  {metric_name.upper()}: {value:.4f}" if isinstance(value, float) else f"  {metric_name.upper()}: {value}")

                              print("="*80)

                              # Also check full report
                              key_full = parts[1] + '/evaluation_report.json'
                              try:
                                  obj_full = s3_client.get_object(Bucket=bucket, Key=key_full)
                                  full_report = json.loads(obj_full['Body'].read())

                                  print("\nQUALITY GATES:")
                                  print("="*80)
                                  qg = full_report.get('quality_gates', {})
                                  print(f"  Passed: {qg.get('passed', 'N/A')}")

                                  for check in qg.get('checks', []):
                                      status = "✓" if check['passed'] else "✗"
                                      print(f"  {status} {check['metric'].upper()}: {check['value']:.4f} (threshold: {check['threshold']:.4f})")

                                  if qg.get('failures'):
                                      print(f"\n  Failures: {', '.join(qg['failures'])}")

                                  print("="*80)

                              except Exception as e:
                                  print(f"Could not read full report: {e}")

                              return property_data

                          except Exception as e:
                              print(f"✗ Could not read evaluation report: {e}")

  # Run for current execution
if 'CURRENT_EXECUTION_ARN' in locals():
      get_actual_metrics(CURRENT_EXECUTION_ARN)
else:
      print("No execution found. Set CURRENT_EXECUTION_ARN first.")

## 6. Utility Functions

Additional helper functions for pipeline management.

In [ ]:
def stop_execution(execution_arn):
    """Stop a running pipeline execution."""
    try:
        response = sagemaker_client.stop_pipeline_execution(
            PipelineExecutionArn=execution_arn
        )
        print(f"✓ Stopped execution: {execution_arn}")
        return response
    except Exception as e:
        print(f"✗ Failed to stop execution: {e}")
        return None

def delete_pipeline(pipeline_name):
    """Delete a pipeline (use with caution!)."""
    try:
        response = sagemaker_client.delete_pipeline(
            PipelineName=pipeline_name
        )
        print(f"✓ Deleted pipeline: {pipeline_name}")
        return response
    except sagemaker_client.exceptions.ResourceNotFound:
        print(f"ℹ Pipeline '{pipeline_name}' does not exist (already deleted or never created)")
        return None
    except Exception as e:
        print(f"✗ Failed to delete pipeline: {e}")
        return None

def get_pipeline_definition(pipeline_name):
    """Get pipeline definition JSON."""
    try:
        response = sagemaker_client.describe_pipeline(
            PipelineName=pipeline_name
        )
        definition = json.loads(response['PipelineDefinition'])
        return definition
    except Exception as e:
        print(f"✗ Failed to get pipeline definition: {e}")
        return None

print("Utility functions loaded:")
print("  - stop_execution(execution_arn)")
print("  - delete_pipeline(pipeline_name)")
print("  - get_pipeline_definition(pipeline_name)")

In [ ]:
import boto3, pandas as pd, io, time

athena_client = boto3.client('athena', region_name=region)
s3_client = boto3.client('s3', region_name=region)

response = athena_client.start_query_execution(
    QueryString="SELECT * FROM training_data LIMIT 3",
    QueryExecutionContext={'Database': 'fraud_detection'},
    ResultConfiguration={'OutputLocation': f's3://{default_bucket}/athena-query-results/'}
)

time.sleep(10)

response = athena_client.get_query_execution(QueryExecutionId=response['QueryExecutionId'])
output_uri = response['QueryExecution']['ResultConfiguration']['OutputLocation']
bucket = output_uri.replace('s3://', '').split('/')[0]
key = '/'.join(output_uri.replace('s3://', '').split('/')[1:])

obj = s3_client.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

print(f"Athena columns: {list(df.columns)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nFirst 2 rows:")
print(df.head(2))

## 7. Quick Reference

### Common Tasks

**Start a new execution with custom parameters:**
```python
PIPELINE_PARAMS['MinRocAuc'] = '0.90'
PIPELINE_PARAMS['EndpointName'] = 'fraud-detector-v2'
# Then run cell 4 to start execution
```

**Stop a running execution:**
```python
stop_execution(CURRENT_EXECUTION_ARN)
```

**Monitor a specific execution:**
```python
execution_arn = 'arn:aws:sagemaker:us-east-1:123456789012:pipeline/fraud-detection-pipeline/execution/xyz'
monitor_execution(execution_arn)
```

**View CloudWatch logs:**
- Go to CloudWatch Console
- Navigate to Log Groups
- Search for `/aws/sagemaker/ProcessingJobs` or `/aws/sagemaker/TrainingJobs`

### Expected Results

With `train.py`, you should see:
- ✓ **ROC-AUC > 0.85** (likely 0.90-0.95)
- ✓ **PR-AUC > 0.50** (likely 0.60-0.80)
- ✓ **True Positives > 0** (model detects fraud)
- ✓ **Quality gates pass**
- ✓ **Pipeline completes successfully**


### Troubleshooting

**Pipeline creation fails:**
- Check that `.env` file is properly configured
- Verify SageMaker execution role has necessary permissions
- Check S3 bucket access

**Training step fails:**
- Check CloudWatch logs for the training job
- Verify input data path is correct
- Check that preprocessing completed successfully

**Evaluation fails quality gates:**
- Check evaluation metrics in cell 6
- If ROC-AUC is still 0.50, verify the pipeline picked up the fixed code
- Consider adjusting quality gate thresholds in PIPELINE_PARAMS